# RDD

In [5]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 84.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 85.7 MB/s  0:00:00
  Attempting uninstall: botocore━━━╸━━━━━━━━━━━━  9/13 [py7zr]todomex]
    Found existing installation: botocore 1.42.30m━━━━━━━━━━━━  9/13 [py7zr]
    Uninstalling botocore-1.42.30:m╸━━━━━━━━━━━━  9/13 [py7zr]
      Successfully uninstalled botocore-1.42.30╸━━━━━━━━━ 10/13 [botocore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [boto3]m12/13 [boto3]re]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 3.1.1 requires botocore<1.42.31,>=1.41.0, but you have botocore 1.42.36 which is incompatible.


### Import données

In [36]:
import importlib
import credentials
importlib.reload(credentials)
import os 
import pandas as pd
from BDTopo_fonctions import load_gpkg

#vote
remote_path="Elections/fichier_vote_avec_code_postal_clean.csv"
local_path = f"/tmp/{os.path.basename(remote_path)}"
credentials.s3.download_file("mgarbe", remote_path, local_path)
df_vote=pd.read_csv(local_path, sep=",")

#code nuances politiques
remote_path="Elections/nuances.csv"
local_path = f"/tmp/{os.path.basename(remote_path)}"
credentials.s3.download_file("mgarbe", remote_path, local_path)
nuances=pd.read_csv(local_path, sep=",")

#données geo
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg") #SANS CLUSTER !!!
gdf=gdf[gdf["Base"]=="Sitadel"].copy()

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Chargement réussi (324979 lignes)


In [38]:
nuance_dict = pd.Series(nuances['Code'].values, index=nuances['Nuance']).to_dict()
df_vote['Nuance_muni'] = df_vote['Nuance_muni'].map(nuance_dict)
df_vote['Nuance_interco'] = df_vote['Nuance_interco'].map(nuance_dict)
df_vote=df_vote[df_vote["Nuance_muni"]!=0].copy()

In [43]:
df_vote = df_vote[(df_vote['voix_pct'] > 50) & (df_vote['voix_pct'] < 100)].copy() #hors triangulaires, 2 candidats
df_vote['delta_score_1'] = df_vote['Nuance_muni'] * (df_vote['voix_pct'] - 50)
df_vote

,annee,ident_election_ville,Nuance_muni,voix_pct,Nuance_interco,delta_score_1
0,2014,01004,1.0,61.309524,NaN,11.309524
3,2014,01010,1.0,56.153846,NaN,6.153846
6,2014,01022,1.0,65.092749,1.0,15.092749
7,2014,01024,1.0,59.154930,-1.0,9.154930
9,2014,01027,1.0,72.077922,1.0,22.077922
...,...,...,...,...,...,...
12256,2020,95598,1.0,50.940212,1.0,0.940212
12258,2020,95607,1.0,58.329564,1.0,8.329564
12260,2020,95637,-1.0,51.761583,-1.0,-1.761583
12261,2020,95652,1.0,73.375796,NaN,23.375796


In [44]:
#on réduit aux communes dans la fenêtre du RDD
seuil=4
df_vote_seuil=df_vote[abs(df_vote["delta_score_1"])<seuil].copy()

In [45]:
id_communes=df_vote_seuil.groupby("annee")["ident_election_ville"].unique()

In [46]:
id_communes

annee
2014    [01034, 01053, 01093, 01143, 01159, 01281, 012...
2020    [01004, 01071, 01160, 01185, 01194, 01249, 012...
Name: ident_election_ville, dtype: object

In [47]:
import geopandas as gpd

# Fonction pour créer la colonne mandat_elec
def assign_mandat(annee):
    if 2014 <= annee <= 2019:
        return 2014  # groupe 1
    elif 2020 <= annee <= 2025:
        return 2020  # groupe 2
    else:
        return None   # pour filtrer les années hors des plages

# Création de la colonne
gdf['mandat_elec'] = gdf['Annee_REF'].apply(assign_mandat)

# Filtrage uniquement sur les périodes souhaitées
gdf = gdf[gdf['mandat_elec'].notna()].copy()

In [48]:
# Normaliser COMM à 5 caractères
gdf['COMM'] = gdf['COMM'].astype(str)
gdf.loc[gdf['COMM'].str.len() == 4, 'COMM'] = '0' + gdf.loc[gdf['COMM'].str.len() == 4, 'COMM']

# Filtrer par mandat_elec et id_communes correspondantes
gdf_RDD_list = []

for mandat in id_communes.index:
    communes = [str(c).zfill(5) for c in id_communes[mandat]]  # s'assure que 5 chiffres
    gdf_RDD_list.append(
        gdf[(gdf['mandat_elec'] == mandat) & (gdf['COMM'].isin(communes))]
    )

# Concaténer les deux groupes
gdf_RDD = pd.concat(gdf_RDD_list, ignore_index=True)

In [49]:
gdf_RDD.groupby("mandat_elec").size()

mandat_elec
2014    1104
2020    1442
dtype: int64

In [50]:
gdf_RDD['COMM'] = gdf_RDD['COMM'].astype(str)
df_vote['ident_election_ville'] = df_vote['ident_election_ville'].astype(str)

# Merge
gdf_RDD = gdf_RDD.merge(
    df_vote[['ident_election_ville', 'Nuance_muni', 'Nuance_interco']],
    left_on='COMM',
    right_on='ident_election_ville',
    how='left'
)

# Optionnel : supprimer la colonne ident_election_ville redondante
gdf_RDD = gdf_RDD.drop(columns=['ident_election_ville']).copy()

In [51]:
nuances

,Nuance,Libellé,Code
0,EXG,Extrême gauche,-1
1,COM,Communiste,-1
2,PG,Parti de Gauche,-1
3,SOC,Socialiste,-1
4,RDG,Radical de Gauche,-1
...,...,...,...
81,LMMD,Liste Majorité-MoDem,1
82,LCEN,Liste d'union du centre,1
83,LDLR,Liste Debout la République,1
84,DLR,Debout la République,1


## RESULTATS 

In [52]:
gdf_RDD.groupby(['mandat_elec','Nuance_muni'])['COMM'].nunique().reset_index(name='nb_communes')

,mandat_elec,Nuance_muni,nb_communes
0,2014,-1.0,150
1,2014,1.0,191
2,2020,-1.0,147
3,2020,1.0,216


Nb permis

In [53]:
obs_par_commune = gdf_RDD.groupby(['Nuance_muni','mandat_elec','COMM']).size().reset_index(name='nb_obs')
RDD_nb_bati = obs_par_commune.groupby(['mandat_elec','Nuance_muni'])['nb_obs'].mean()
RDD_nb_bati=RDD_nb_bati.reset_index()
print(RDD_nb_bati)


   mandat_elec  Nuance_muni    nb_obs
0         2014         -1.0  4.966667
1         2014          1.0  5.047120
2         2020         -1.0  5.265306
3         2020          1.0  5.217593


In [54]:
from scipy import stats

mandats = RDD_nb_bati['mandat_elec'].unique()

for mandat in mandats:
    # Sélection des communes pour ce mandat
    df_m = gdf_RDD[gdf_RDD['mandat_elec'] == mandat]
    
    # Extraire SURF_CREEE par groupe Nuance_muni
    groupe_neg1 = df_m[df_m['Nuance_muni'] == -1].groupby('COMM').size()
    groupe_pos1 = df_m[df_m['Nuance_muni'] == 1].groupby('COMM').size()
    
    # Test t de Welch (variances inégales)
    t_stat, p_val = stats.ttest_ind(groupe_neg1, groupe_pos1, equal_var=False)
    
    print(f"Mandat {mandat}: t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.1:
        print("→ Différence significative de surface moyenne entre les groupes")
    else:
        print("→ Pas de différence significative de surface moyenne entre les groupes")

Mandat 2014: t = -0.126, p = 0.8998
→ Pas de différence significative de surface moyenne entre les groupes
Mandat 2020: t = 0.071, p = 0.9435
→ Pas de différence significative de surface moyenne entre les groupes


Surface créee

In [55]:
surf_par_commune = gdf_RDD.groupby(['Nuance_muni', 'mandat_elec', 'COMM'])['SURF_CREEE'].sum().reset_index(name='surf_commune')
RDD_surf = surf_par_commune.groupby(['mandat_elec','Nuance_muni'])['surf_commune'].mean().reset_index()

print(RDD_surf)

   mandat_elec  Nuance_muni  surf_commune
0         2014         -1.0  25049.220000
1         2014          1.0  21340.225131
2         2020         -1.0  23862.578231
3         2020          1.0  21264.287037


In [56]:
surf_par_commune

,Nuance_muni,mandat_elec,COMM,surf_commune
0,-1.0,2014,03025,1145.0
1,-1.0,2014,03093,2200.0
2,-1.0,2014,03264,4714.0
3,-1.0,2014,04197,1056.0
4,-1.0,2014,05023,3499.0
...,...,...,...,...
699,1.0,2020,94071,49160.0
700,1.0,2020,94074,20927.0
701,1.0,2020,95134,10601.0
702,1.0,2020,95268,3467.0


In [57]:
mandats = RDD_surf['mandat_elec'].unique()

for mandat in mandats:
    # Sélection des communes pour ce mandat
    df_m = gdf_RDD[gdf_RDD['mandat_elec'] == mandat]
    
    # Extraire SURF_CREEE par groupe Nuance_muni
    groupe_neg1 = df_m[df_m['Nuance_muni'] == -1].groupby('COMM')['SURF_CREEE'].sum()
    groupe_pos1 = df_m[df_m['Nuance_muni'] == 1].groupby('COMM')['SURF_CREEE'].sum()
    
    # Test t VOIR WELCH VS STUDENT
    t_stat, p_val = stats.ttest_ind(groupe_neg1, groupe_pos1, equal_var=True) #student
    
    print(f"Mandat {mandat}: t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.1:
        print("→ Différence significative de surface moyenne entre les groupes")
    else:
        print("→ Pas de différence significative de surface moyenne entre les groupes")

Mandat 2014: t = 0.673, p = 0.5015
→ Pas de différence significative de surface moyenne entre les groupes
Mandat 2020: t = 0.599, p = 0.5496
→ Pas de différence significative de surface moyenne entre les groupes


In [58]:
from rdd import rdd

mandats = df_RDD_surfrdd['mandat_elec'].unique()

for mandat in mandats:
    df_m = RDD_surf[RDD_surf['mandat_elec'] == mandat]

    # Définir forcing variable et outcome
    forcing = df_m['score_margin'].values
    outcome = df_m['surf_commune'].values

    # Créer RDD
    rdd_model = rdd.RDD(outcome, forcing, c=0)  # seuil à 0

    # Ajustement local linear (kernel triangulaire)
    rdd_model.fit(kernel='triangular', p=1)

    # Résumé
    print(f"\nMandat {mandat}")
    print(rdd_model.summary())

NameError: name 'df_RDD_surfrdd' is not defined

Surface moyenne par bat

In [59]:
surf_moyenne = gdf_RDD.groupby(['mandat_elec','Nuance_muni'])['SURF_CREEE'].mean().reset_index(name='surf_moy')
print(surf_moyenne)

   mandat_elec  Nuance_muni     surf_moy
0         2014         -1.0  5043.467114
1         2014          1.0  4228.198133
2         2020         -1.0  4532.040052
3         2020          1.0  4075.497782


In [24]:
mandats = RDD_surf['mandat_elec'].unique()

for mandat in mandats:
    # Sélection des communes pour ce mandat
    df_m = gdf_RDD[gdf_RDD['mandat_elec'] == mandat]
    
    # Extraire SURF_CREEE par groupe Nuance_muni
    groupe_neg1 = df_m[df_m['Nuance_muni'] == -1]['SURF_CREEE']
    groupe_pos1 = df_m[df_m['Nuance_muni'] == 1]['SURF_CREEE']
    
    # Test t de Welch (variances inégales)
    t_stat, p_val = stats.ttest_ind(groupe_neg1, groupe_pos1, equal_var=False)
    
    print(f"Mandat {mandat}: t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.1:
        print("→ Différence significative de surface moyenne entre les groupes")
    else:
        print("→ Pas de différence significative de surface moyenne entre les groupes")

Mandat 2014: t = 4.461, p = 0.0000
→ Différence significative de surface moyenne entre les groupes
Mandat 2020: t = 2.061, p = 0.0394
→ Différence significative de surface moyenne entre les groupes
